In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from scipy.spatial.distance import cdist, pdist
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from threadpoolctl import threadpool_limits

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/pitching'
OUT = ROOT / 'results/fastball_delivery_styles'
FIG = ROOT / 'figures/fastball_delivery_styles'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)

AUTO_DOWNLOAD = True
SEED = 2108
N_RESAMPLES = 150
N_OUTCOME_DRAWS = 2000
MIN_PITCHES = 3
PHASE_STEPS = 25
FIXED_MS = 70
MATCH_MPH = 1.0
K_VALUES = (2, 3, 4, 5)
MAX_COCKING_MS = 300
MAX_ACCELERATION_MS = 80
EVENTS = ['fp_100_time', 'MER_time', 'BR_time']
SIGNALS = {
    'pelvis_angle_z': 'Pelvis rotation',
    'torso_angle_z': 'Torso rotation',
    'torso_angle_x': 'Trunk forward tilt',
    'lead_knee_angle_x': 'Lead-knee flexion',
}
ANGLE_COLS = list(SIGNALS)
ARCHIVES = {
    'joint_angles': 'b4c4854ffcc52a0f757e0b3a3c86acdc4a2fc2e50516a76e62aabdb916afed46',
}
PHASE_GRID = np.linspace(0, 2, 2 * PHASE_STEPS + 1)
FIXED_GRID_MS = np.linspace(-FIXED_MS, 0, len(PHASE_GRID))

print('Study 21 | Fastball Delivery Styles')
print('Dataset: OpenBiomechanics Project, Dataset v1. Kinematics from foot plant to release.')
print('Setup: numpy, pandas, scipy, matplotlib, scikit-learn, threadpoolctl.')
print('First run downloads the official angle archive and two CSVs. Later runs use local data.')

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, path):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {path}; set AUTO_DOWNLOAD=True or supply the official file.')
    print(f'Downloading {path.name} ...', flush=True)
    temporary = path.with_suffix(path.suffix + '.part')
    urlretrieve(url, temporary)
    temporary.replace(path)

base = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
csv_base = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_pitching/data/'
for name, digest in ARCHIVES.items():
    target = RAW / (name + '.csv')
    if not target.exists():
        archive = RAW / ('pitching_' + name + '.zip')
        if not archive.exists():
            download(base + archive.name, archive)
        if sha256(archive) != digest:
            raise ValueError(f'{archive.name}: Dataset v1 archive hash mismatch')
        with zipfile.ZipFile(archive) as z:
            matches = [n for n in z.namelist() if Path(n).name == target.name]
            if len(matches) != 1:
                raise ValueError(f'Expected exactly one {target.name}')
            temporary = target.with_suffix('.csv.part')
            with z.open(matches[0]) as source, temporary.open('wb') as dest:
                shutil.copyfileobj(source, dest)
            temporary.replace(target)
for name, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv')]:
    if not (RAW / name).exists():
        download(csv_base + remote, RAW / name)

metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_pitch': str, 'user': str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_pitch': str})
angles = pd.read_csv(RAW / 'joint_angles.csv',
                     usecols=['session_pitch', 'time'] + EVENTS + ANGLE_COLS,
                     dtype={'session_pitch': str})
for name, table in [('metadata', metadata), ('POI', poi)]:
    if table.session_pitch.isna().any() or table.session_pitch.duplicated().any():
        raise ValueError(f'Missing or duplicate pitch IDs in {name}')
if angles.duplicated(['session_pitch', 'time']).any():
    raise ValueError('Duplicate pitch/time pairs in joint angles')

pitches = poi.loc[poi.pitch_type.eq('FF'), ['session_pitch', 'p_throws', 'pitch_speed_mph']].merge(
    metadata[['session_pitch', 'user', 'session_mass_kg', 'session_height_m', 'playing_level']],
    on='session_pitch', how='left', validate='one_to_one')
angle_groups = {key: g.sort_values('time') for key, g in angles.groupby('session_pitch', sort=False)}
manifest = {name: sha256(RAW / name) for name in ('joint_angles.csv', 'metadata.csv', 'poi_metrics.csv')}
(OUT / 'input_hashes.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

In [ ]:
def phase_window(g, columns, lo, hi):
    if g is None or not np.isfinite([lo, hi]).all() or lo >= hi:
        raise ValueError('missing signal or invalid phase boundaries')
    t = g.time.to_numpy(float)
    if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid time grid')
    if lo < t[0] or hi > t[-1]:
        raise ValueError('phase outside sampled recording')
    dt = np.median(np.diff(t))
    if not np.allclose(np.diff(t), dt, atol=0.00011, rtol=0.03):
        raise ValueError('irregular sampling')
    left = max(0, np.searchsorted(t, lo, side='right') - 1)
    right = min(len(t), np.searchsorted(t, hi, side='left') + 1)
    tt, values = t[left:right], g[columns].to_numpy(float)[left:right]
    if len(tt) < 2 or not np.isfinite(values).all():
        raise ValueError('missing phase samples or signal gaps')
    for i, col in enumerate(columns):
        if col in ('pelvis_angle_z', 'torso_angle_z'):
            values[:, i] = np.rad2deg(np.unwrap(np.deg2rad(values[:, i])))
    grid = np.r_[lo, tt[(tt > lo) & (tt < hi)], hi]
    sampled = np.column_stack([np.interp(grid, tt, values[:, i]) for i in range(len(columns))])
    return grid, sampled

def event_values(g):
    if g is None:
        raise ValueError('missing angle signal')
    values = {}
    for name in EVENTS:
        if g[name].nunique(dropna=False) != 1:
            raise ValueError('inconsistent event annotation')
        values[name] = float(g[name].iloc[0])
    return values


def interpolate_curves(t, values, grid):
    return np.column_stack([np.interp(grid, t, channel) for channel in values.T])

audit_rows, records, aligned_curves, fixed_curves = [], [], {}, {}
for pitch in pitches.itertuples(index=False):
    key = pitch.session_pitch
    status = 'retained'
    try:
        if pd.isna(pitch.user):
            raise ValueError('missing pitcher ID')
        group = angle_groups.get(key)
        events = event_values(group)
        plant, mer, release = (events[name] for name in EVENTS)
        if not np.isfinite([plant, mer, release]).all() or not plant < mer < release:
            raise ValueError('missing or unordered plant, MER, or release')
        t, values = phase_window(group, ANGLE_COLS, plant, release)
        phase_t = np.r_[np.linspace(plant, mer, PHASE_STEPS + 1),
                        np.linspace(mer, release, PHASE_STEPS + 1)[1:]]
        aligned_curves[key] = interpolate_curves(t, values, phase_t)
        has_fixed = release + FIXED_GRID_MS[0] / 1000 >= plant
        if has_fixed:
            fixed_curves[key] = interpolate_curves(t, values, release + FIXED_GRID_MS / 1000)
        records.append({**pitch._asdict(), 'cocking_ms': (mer - plant) * 1000,
                        'acceleration_ms': (release - mer) * 1000, 'fixed_window': has_fixed})
    except ValueError as exc:
        status = str(exc)
    audit_rows.append({'session_pitch': key, 'pitcher_id': pitch.user, 'status': status})

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUT / 'pitch_audit.csv', index=False)
display(audit.status.value_counts().rename('fastballs').to_frame())
if not records:
    raise ValueError('No valid pitch trajectories. See pitch_audit.csv.')
trials = pd.DataFrame(records)
trials.to_csv(OUT / 'retained_fastballs.csv', index=False)

def pitcher_reps(table, curves):
    return {pitcher: np.stack([curves[key] for key in rows.session_pitch])
            for pitcher, rows in table.groupby('user', sort=True) if len(rows) >= MIN_PITCHES}

reps = pitcher_reps(trials, aligned_curves)
ids = np.array(list(reps))
if len(ids) < 15:
    raise ValueError(f'Only {len(ids)} eligible pitchers. Check the pitch audit and minimum pitch count.')
eligible = trials[trials.user.isin(ids)]
print(f'{len(trials)} valid fastballs. Primary analysis: {len(eligible)} fastballs from {len(ids)} pitchers.')
print(f'At least {MIN_PITCHES} valid fastballs per pitcher. Movement coordinates follow OBP conventions.')

In [ ]:
def median_profiles(repetitions, rng=None):
    return np.stack([np.median(rows if rng is None else rows[rng.integers(len(rows), size=len(rows))],
                               axis=0) for rows in repetitions.values()])

def fit_pca(profiles):
    mean = profiles.mean(axis=0)
    scale = np.sqrt(np.mean((profiles - mean) ** 2, axis=(0, 1)))
    scale = np.where(scale > 1e-9, scale, 1.0)
    matrix = ((profiles - mean) / scale).reshape(len(profiles), -1)
    model = PCA(svd_solver='full')
    scores = model.fit_transform(matrix)
    cumulative = np.cumsum(model.explained_variance_ratio_)
    n_components = int(np.searchsorted(cumulative, .85) + 1)
    n_components = min(max(2, n_components), 10, scores.shape[1])
    return model, scores[:, :n_components], scale

def partition(scores, k):
    return KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(scores)

profiles = median_profiles(reps)
fixed_trials = eligible[eligible.fixed_window]
fixed_reps = pitcher_reps(fixed_trials, fixed_curves)
paired_reps = pitcher_reps(fixed_trials, aligned_curves)
matched_indices = np.flatnonzero(np.isin(ids, list(fixed_reps)))
typical_trials = eligible[(eligible.cocking_ms <= MAX_COCKING_MS) &
                          (eligible.acceleration_ms <= MAX_ACCELERATION_MS)]
typical_reps = pitcher_reps(typical_trials, aligned_curves)
typical_indices = np.flatnonzero(np.isin(ids, list(typical_reps)))

with threadpool_limits(limits=1):
    pca, scores, scales = fit_pca(profiles)
    if not np.isfinite(pca.explained_variance_ratio_).all():
        raise ValueError('No measurable between-pitcher trajectory variation.')
    full_labels = {k: partition(scores, k) for k in K_VALUES}
    if len(fixed_reps) >= 15:
        fixed_scores = fit_pca(median_profiles(fixed_reps))[1]
        paired_scores = fit_pca(median_profiles(paired_reps))[1]
        fixed_labels = {k: partition(fixed_scores, k) for k in K_VALUES}
        paired_labels = {k: partition(paired_scores, k) for k in K_VALUES}
    else:
        fixed_labels, paired_labels = {}, {}

    typical_labels = {}
    if len(typical_reps) >= 15:
        typical_scores = fit_pca(median_profiles(typical_reps))[1]
        typical_labels = {k: partition(typical_scores, k) for k in K_VALUES}

    rng = np.random.default_rng(SEED)
    resamples = {k: [] for k in K_VALUES}
    for _ in range(N_RESAMPLES):
        resampled_scores = fit_pca(median_profiles(reps, rng))[1]
        for k in K_VALUES:
            resamples[k].append(partition(resampled_scores, k))

rows = []
for k, labels in full_labels.items():
    ari = [adjusted_rand_score(labels, other) for other in resamples[k]]
    size = np.bincount(labels, minlength=k)
    rows.append(dict(k=k, silhouette=silhouette_score(scores, labels) if (size > 0).all() else np.nan,
                     smallest_group=int(size.min()), resample_ari_p10=np.quantile(ari, .10),
                     resample_ari_median=np.median(ari),
                     alignment_ari=adjusted_rand_score(paired_labels[k], fixed_labels[k])
                     if k in fixed_labels else np.nan,
                     cohort_ari=adjusted_rand_score(labels[matched_indices], paired_labels[k])
                     if k in paired_labels else np.nan,
                     duration_ari=adjusted_rand_score(labels[typical_indices], typical_labels[k])
                     if k in typical_labels else np.nan))
checks = pd.DataFrame(rows).set_index('k')
checks['passes_screen'] = (
    checks.silhouette.ge(.30) & checks.smallest_group.ge(max(8, int(np.ceil(.10 * len(ids))))) &
    checks.resample_ari_median.ge(.70) & checks.resample_ari_p10.ge(.50) &
    checks.alignment_ari.ge(.60) & checks.cohort_ari.ge(.60) & checks.duration_ari.ge(.60))
checks.to_csv(OUT / 'cluster_diagnostics.csv')
pd.DataFrame({'component': np.arange(1, len(pca.explained_variance_ratio_) + 1),
              'variance_explained': pca.explained_variance_ratio_}).to_csv(OUT / 'pca_variance.csv', index=False)
passed = checks[checks.passes_screen]
selected_k = int(passed.silhouette.idxmax()) if len(passed) else None
candidate_k = int(checks.silhouette.idxmax()) if checks.silhouette.notna().any() else None


In [ ]:
def unique_context(values):
    observed = sorted(values.dropna().astype(str).unique())
    return '/'.join(observed) if observed else 'Unknown'

pitcher_table = eligible.groupby('user').agg(
    n_pitches=('session_pitch', 'size'), velocity_mph=('pitch_speed_mph', 'mean'),
    n_velocities=('pitch_speed_mph', 'count'), throws=('p_throws', unique_context),
    playing_level=('playing_level', unique_context), height_m=('session_height_m', 'median'),
    mass_kg=('session_mass_kg', 'median'), cocking_ms=('cocking_ms', 'median'),
    acceleration_ms=('acceleration_ms', 'median')).reindex(ids)
pitcher_table.index.name = 'pitcher_id'
for j, axis in enumerate(scores.T, 1):
    pitcher_table[f'PC{j}'] = axis
for k, labels in full_labels.items():
    pitcher_table[f'candidate_group_k{k}'] = labels + 1
if selected_k:
    pitcher_table['group'] = full_labels[selected_k] + 1
pitcher_table.to_csv(OUT / 'pitcher_profiles.csv')



In [ ]:
from itertools import combinations

i, j = np.triu_indices(len(ids), k=1)
mph = pitcher_table.velocity_mph.to_numpy()
gaps = np.abs(mph[i] - mph[j])
distance = pdist(scores)
typical_pitchers = pitcher_table.cocking_ms.le(MAX_COCKING_MS).to_numpy() & (
    pitcher_table.acceleration_ms.le(MAX_ACCELERATION_MS).to_numpy())
match = np.flatnonzero((gaps <= MATCH_MPH) & typical_pitchers[i] & typical_pitchers[j])

def project_profiles(curves):
    matrix = ((curves - profiles.mean(axis=0)) / scales).reshape(len(curves), -1)
    return pca.transform(matrix)[:, :scores.shape[1]]

def split_rep_distances(rows):
    half = len(rows) // 2
    subsets = [left for left in combinations(range(len(rows)), half)
               if len(rows) % 2 or 0 in left]
    partitions = np.zeros((len(subsets), len(rows)), dtype=bool)
    for mask, subset in zip(partitions, subsets):
        mask[list(subset)] = True
    left = project_profiles(np.stack([np.median(rows[mask], axis=0) for mask in partitions]))
    right = project_profiles(np.stack([np.median(rows[~mask], axis=0) for mask in partitions]))
    return np.linalg.norm(left - right, axis=1)

within = [split_rep_distances(reps[pitcher]) for pitcher in ids]
within_95 = np.array([np.quantile(distances, .95) for distances in within])
within_table = pd.DataFrame({
    'pitcher_id': ids,
    'splits': [len(distances) for distances in within],
    'split_distance_median': [np.median(distances) for distances in within],
    'split_distance_p95': within_95,
})
within_table.to_csv(OUT / 'within_pitcher_split_distances.csv', index=False)

pairs = pd.DataFrame({
    'pitcher_a': ids[i], 'pitcher_b': ids[j],
    'velocity_gap_mph': gaps, 'delivery_distance': distance,
    'larger_within_p95': np.maximum(within_95[i], within_95[j]),
})
pairs['above_both_split_benchmarks'] = pairs.delivery_distance > pairs.larger_within_p95
pairs['both_typical_phase_medians'] = typical_pitchers[i] & typical_pitchers[j]

def summarize_pairs(table):
    by_pitcher = pd.concat([
        table[['pitcher_a', 'above_both_split_benchmarks']].rename(columns={'pitcher_a': 'pitcher_id'}),
        table[['pitcher_b', 'above_both_split_benchmarks']].rename(columns={'pitcher_b': 'pitcher_id'}),
    ]).groupby('pitcher_id').above_both_split_benchmarks.agg(['size', 'mean'])
    by_pitcher.columns = ['matched_pitchers', 'share_above_both_benchmarks']
    return by_pitcher

matched = pairs.loc[gaps <= MATCH_MPH].copy()
matched.to_csv(OUT / 'velocity_matched_pairs.csv', index=False)
pitcher_matched = summarize_pairs(matched)
pitcher_matched.to_csv(OUT / 'velocity_matches_by_pitcher.csv')

sensitivity = []
for tolerance, typical_only in [(MATCH_MPH, False), (.5, False), (2., False), (MATCH_MPH, True)]:
    subset = pairs[pairs.velocity_gap_mph.le(tolerance)]
    if typical_only:
        subset = subset[subset.both_typical_phase_medians]
    by_pitcher = summarize_pairs(subset)
    sensitivity.append(dict(velocity_tolerance_mph=tolerance, typical_phase_medians_only=typical_only,
                            pairs=len(subset), pitchers=len(by_pitcher),
                            pitcher_weighted_share_above=by_pitcher.share_above_both_benchmarks.mean()))
sensitivity = pd.DataFrame(sensitivity)
sensitivity.to_csv(OUT / 'matched_velocity_sensitivity.csv', index=False)

print('Similar Velocity, Different Deliveries')
print('Within-pitcher benchmark: distance between median profiles from disjoint subsets of the same pitcher’s reps.')
print('All distances use the same fitted PCA and scaling. Split benchmarks use fewer reps than full profiles.')
print('A pair clears the benchmark if its distance exceeds both pitchers’ 95th-percentile split distances.')
print('Shares weight each pitcher with a match equally. Shared pitchers make pair results dependent.')
print('The split benchmark is descriptive. With few reps per pitcher, its upper percentile can be noisy.')
display(sensitivity.round(3))

if len(matched):
    def ecdf(ax, values, weights, label):
        order = np.argsort(values)
        x = np.asarray(values)[order]
        y = np.cumsum(np.asarray(weights)[order]) / np.sum(weights)
        ax.step(np.r_[x[0], x], np.r_[0., y], where='post', label=label)

    counts = pitcher_matched.matched_pitchers
    pair_weights = 1 / matched.pitcher_a.map(counts) + 1 / matched.pitcher_b.map(counts)
    matched_within = [within[index] for index, pitcher in enumerate(ids) if pitcher in counts.index]
    split_weights = np.concatenate([np.full(len(values), 1 / len(values)) for values in matched_within])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ecdf(ax, np.concatenate(matched_within), split_weights, 'Within pitcher (disjoint rep splits)')
    ecdf(ax, matched.delivery_distance, pair_weights, f'Between pitchers (within {MATCH_MPH:g} mph)')
    if len(match):
        example_distance = distance[match[np.argmax(distance[match])]]
        percentile = 100 * np.average(matched.delivery_distance.le(example_distance), weights=pair_weights)
        ax.axvline(example_distance, color='0.35', linestyle='--', label='Illustrated pair')
        print(f'The illustrated pair is at the {percentile:.1f}th percentile of matched delivery distance '
              'with equal pitcher weighting. It was chosen to show a large contrast.')
    ax.set(xlabel='Distance in retained PCA space', ylabel='Cumulative share',
           title='Delivery distances within pitchers and between similar-speed pitchers')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(FIG / 'matched_velocity_distance_distribution.png', dpi=180, bbox_inches='tight')
    plt.show()


In [ ]:
def plot_reps(pitcher_ids, title, filename):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    for j, (ax, signal_title) in enumerate(zip(axes.flat, SIGNALS.values())):
        for color, pitcher in enumerate(pitcher_ids):
            i = np.flatnonzero(ids == pitcher)[0]
            mph = pitcher_table.loc[pitcher, 'velocity_mph']
            speed = f'{mph:.1f} mph' if np.isfinite(mph) else 'velocity unavailable'
            ax.plot(PHASE_GRID, reps[pitcher][:, :, j].T, color=f'C{color}', alpha=.15)
            ax.plot(PHASE_GRID, profiles[i, :, j], color=f'C{color}', linewidth=2.3,
                    label=f'{pitcher} · {speed}')
        ax.axvline(1, color='0.6', linewidth=.8, linestyle='--')
        ax.set(title=signal_title, ylabel='OBP angle (°)', xticks=[0, 1, 2],
               xticklabels=['Plant', 'MER', 'Release'])
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(FIG / filename, dpi=180, bbox_inches='tight')
    plt.show()

if len(match):
    best = match[np.argmax(distance[match])]
    pair = [ids[i[best]], ids[j[best]]]
    pitcher_table.loc[pair].to_csv(OUT / 'matched_velocity_example.csv')
    print(f'Illustrative pair selected for large movement distance within {MATCH_MPH:g} mph.')
    display(pitcher_table.loc[pair, ['velocity_mph', 'n_pitches', 'cocking_ms', 'acceleration_ms']].round(2))
    print(f'Velocity difference: {gaps[best]:.2f} mph. PCA distance: {distance[best]:.2f}.')
    plot_reps(pair, 'Similar velocity, different deliveries', 'matched_velocity_deliveries.png')
else:
    print(f'No pair within the phase duration limits had mean velocities within {MATCH_MPH:g} mph.')

if selected_k:
    examples = []
    for group in np.unique(full_labels[selected_k]):
        indices = np.flatnonzero(full_labels[selected_k] == group)
        group_scores = scores[indices]
        representative = indices[np.argmin(cdist(group_scores, group_scores.mean(axis=0, keepdims=True)))]
        examples.append(ids[representative])
    pitcher_table.loc[examples].to_csv(OUT / 'group_representatives.csv')
    plot_reps(examples, 'Representative pitchers from supported delivery groups', 'group_representatives.png')

print('The sample is mostly collegiate and captures one session per pitcher.')
print('Group stability describes repeated reps in this dataset. Velocity associations are descriptive.')
print(f'Results: {OUT}')
print(f'Figures: {FIG}')

In [ ]:
print(f'{scores.shape[1]} retained PCs explain {pca.explained_variance_ratio_[:scores.shape[1]].sum():.1%} of trajectory variance.')
print(f'Alignment check uses {len(fixed_reps)} pitchers with at least {MIN_PITCHES} matched pitches covering the final {FIXED_MS} ms.')
print('Screens are exploratory. ARI measures agreement after accounting for chance.')
print(f'Duration check uses {len(typical_reps)} pitchers after excluding '
      f'{len(eligible) - len(typical_trials)} pitches above Study 08’s cocking or acceleration limits.')
with pd.option_context('display.max_columns', None):
    display(checks.round(3))
print(f'Supported candidate: {selected_k} groups.' if selected_k else
      'No candidate met the separation and stability screens. Interpret the continuous PCA profiles.')

In [ ]:
if candidate_k:
    labels = full_labels[candidate_k] + 1
    repetitions = np.stack(resamples[candidate_k])
    coassignment = (repetitions[:, :, None] == repetitions[:, None, :]).mean(axis=0)
    pd.DataFrame(coassignment, index=ids, columns=ids).rename_axis('pitcher_id').to_csv(
        OUT / 'candidate_coassignment.csv')
    context = pitcher_table.assign(candidate_group=labels)
    print(f'Context for the best silhouette candidate (k={candidate_k}).')
    for col in ('throws', 'playing_level'):
        table = pd.crosstab(context.candidate_group, context[col])
        table.to_csv(OUT / f'candidate_{col}.csv')
        display(table)
    table = context.groupby('candidate_group')[['height_m', 'mass_kg', 'cocking_ms', 'acceleration_ms']].median()
    table.to_csv(OUT / 'candidate_context_medians.csv')
    display(table.round(3))

fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for j, (ax, title) in enumerate(zip(axes.flat, SIGNALS.values())):
    for pc in (0, 1):
        mode = pca.components_[pc].reshape(profiles.shape[1:])[:, j] * scales[j]
        delta = np.sqrt(pca.explained_variance_[pc]) * mode
        ax.plot(PHASE_GRID, delta, label=f'PC{pc + 1}: +1 SD from mean')
    ax.axhline(0, color='0.5', linewidth=.7)
    ax.axvline(1, color='0.75', linewidth=.8)
    ax.set(title=title, ylabel='Change from mean (°)', xticks=[0, 1, 2],
           xticklabels=['Plant', 'MER', 'Release'])
axes[0, 0].legend(fontsize=8)
fig.suptitle('How the two leading movement axes change the joint-angle curves')
fig.tight_layout()
fig.savefig(FIG / 'pca_movement_axes.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
rng = np.random.default_rng(SEED + 1)
outcomes = pitcher_table.dropna(subset=['velocity_mph'])

def percentile_interval(values):
    return np.quantile(values, [.025, .975])

pc_results = []
for pc in ('PC1', 'PC2'):
    values = outcomes[[pc, 'velocity_mph']].to_numpy()
    if len(values) < 3 or np.unique(values[:, 1]).size < 2:
        continue
    rho = spearmanr(values[:, 0], values[:, 1]).statistic
    samples = values[rng.integers(len(values), size=(N_OUTCOME_DRAWS, len(values)))]
    rank_x = pd.DataFrame(samples[:, :, 0]).rank(axis=1).to_numpy()
    rank_y = pd.DataFrame(samples[:, :, 1]).rank(axis=1).to_numpy()
    rank_x -= rank_x.mean(axis=1, keepdims=True)
    rank_y -= rank_y.mean(axis=1, keepdims=True)
    denom = np.linalg.norm(rank_x, axis=1) * np.linalg.norm(rank_y, axis=1)
    valid = denom > 0
    draws = np.sum(rank_x[valid] * rank_y[valid], axis=1) / denom[valid]
    lo, hi = percentile_interval(draws) if len(draws) else (np.nan, np.nan)
    pc_results.append(dict(axis=pc, pitchers=len(values), spearman_rho=rho, ci_low=lo, ci_high=hi))
pc_results = pd.DataFrame(pc_results, columns=['axis', 'pitchers', 'spearman_rho', 'ci_low', 'ci_high'])
pc_results.to_csv(OUT / 'velocity_by_pc.csv', index=False)
display(pc_results.round(3))
print('Velocity intervals resample pitchers with the fitted movement representation held fixed.')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
if selected_k:
    for label, rows in pitcher_table.groupby('group'):
        axes[0].scatter(rows.PC1, rows.PC2, label=f'Group {label}', alpha=.8)
    axes[0].legend()
else:
    axes[0].scatter(pitcher_table.PC1, pitcher_table.PC2, alpha=.8)
axes[0].set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
            ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', title='Fastball delivery map')
for ax, pc in zip(axes[1:], ('PC1', 'PC2')):
    ax.scatter(outcomes[pc], outcomes.velocity_mph, alpha=.75)
    ax.set(xlabel=pc, ylabel='Pitcher mean velocity (mph)', title=f'Velocity along {pc}')
fig.tight_layout()
fig.savefig(FIG / 'delivery_map_and_velocity.png', dpi=160, bbox_inches='tight')
plt.show()

if selected_k:
    fig, ax = plt.subplots(figsize=(7, 4))
    summary = []
    for label, group in pitcher_table.groupby('group'):
        mph = group.velocity_mph.dropna().to_numpy()
        lo = hi = np.nan
        if len(mph) >= 2:
            draws = mph[rng.integers(len(mph), size=(N_OUTCOME_DRAWS, len(mph)))].mean(axis=1)
            lo, hi = percentile_interval(draws)
            ax.vlines(label, lo, hi, color='black', zorder=3)
        mean = mph.mean() if len(mph) else np.nan
        summary.append(dict(group=label, pitchers=len(group), pitchers_with_velocity=len(mph),
                            mean_mph=mean, ci_low=lo, ci_high=hi))
        ax.scatter(label + rng.uniform(-.14, .14, len(mph)), mph, alpha=.55)
        ax.scatter(label, mean, marker='D', color='black', zorder=4)
    summary = pd.DataFrame(summary)
    summary.to_csv(OUT / 'velocity_by_group.csv', index=False)
    display(summary.round(2))
    ax.set(xlabel='Delivery group', ylabel='Pitcher mean velocity (mph)',
           xticks=summary.group, title='Velocity by delivery group with 95% bootstrap intervals')
    fig.tight_layout()
    fig.savefig(FIG / 'velocity_by_group.png', dpi=160, bbox_inches='tight')
    plt.show()